# GCM伽罗华计数器模式原理与实现

**摘要：** 详解GCM认证加密模式的原理、特性、安全优势及Python实现，涵盖CTR加密与Galois域认证核心逻辑

- **类别：** 现代密码学
- **难度：** 进阶
- **预计用时：** 3小时
- **先修：** CTR模式基础、伽罗华域GF(2^128)概念、认证加密AEAD、Nonce使用规范
- **学习目标：** 掌握GCM模式的认证加密核心原理与实现

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **认证加密 AEAD：** 同时提供机密性和完整性认证
- **并行处理 Parallel：** 加密和认证均可并行执行
- **AAD支持 AAD：** 支持附加认证数据验证

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 理解GCM模式核心工作原理

GCM（Galois/Counter Mode，伽罗华计数器模式）是目前最主流的**认证加密（AEAD）** 模式，核心逻辑是**CTR模式加密 + Galois域哈希认证** 的结合：
1. 双核心架构：
   a. **CTR加密层**：基于计数器模式生成密钥流，与明文异或得到密文（保证机密性）；
   b. **GHASH认证层**：在伽罗华域GF(2^128)上对密文、附加认证数据（AAD）进行多项式哈希，生成认证标签（保证完整性和真实性）；
2. 完整流程：
   a. 初始化：生成96位推荐长度的Nonce，计算哈希密钥H=E_K(0^128)（加密零块）；
   b. 计数器初始化：96位Nonce拼接00000001得到初始计数器J0；
   c. CTR加密：用递增计数器生成密钥流，异或明文得到密文；
   d. GHASH认证：计算AAD、密文及其长度的伽罗华哈希值S；
   e. 标签生成：S ⊕ E_K(J0)得到128位认证标签；
   f. 解密验证：先验证标签一致性，再执行CTR解密（标签不匹配则直接失败）。
GCM的核心创新是将流式加密与代数认证结合，成为TLS 1.3、IPsec等标准协议的首选模式。


> 提示：GCM的“认证优先”原则：解密时必须先验证标签，标签不匹配则立即终止解密，防止padding oracle等攻击

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 分析GCM模式的实现特性

基于CTR加密+Galois认证的双核心架构，GCM具备以下核心实现特性：
1. Nonce规范：
   - 推荐使用96位（12字节）随机Nonce（NIST标准），可直接拼接计数器；
   - 非96位Nonce需先通过GHASH处理，效率略低；
   - 同一密钥下Nonce**绝对不能重复**，否则认证标签可被伪造；
2. 并行能力：
   - CTR加密：计数器可并行生成，支持并行异或；
   - GHASH认证：分组可并行计算伽罗华乘法，无需串行；
3. 无填充特性：继承CTR模式优势，支持任意长度数据，无需PKCS#7填充；
4. AAD支持：
   - 附加认证数据（如协议头、元数据）仅参与认证不参与加密；
   - 解密时AAD必须与加密时完全一致，否则标签验证失败；
5. 标签特性：
   - 标准标签长度128位（16字节），可按需缩短至96/64/32位（安全性降低）；
   - 标签验证需使用**常数时间比较**，防止计时攻击。
数学核心：GHASH基于GF(2^128)域的多项式乘法，不可约多项式为$x^{128}+x^7+x^2+x+1$。


> 提示：96位Nonce是GCM的最优选择：既保证随机性，又避免额外的GHASH计算，性能最优

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 掌握GCM模式的安全优势与注意事项

### 核心安全优势
1. 认证加密一体化：同时提供机密性（CTR）、完整性（GHASH）、真实性（标签验证），抵御篡改和重放攻击；
2. 标准化认证：符合NIST SP 800-38D、ISO/IEC 19772标准，经过严格密码学分析；
3. 高性能特性：全并行处理能力，硬件实现效率接近AES理论上限；
4. 灵活的AAD支持：适配网络协议中“数据加密、头认证”的常见需求；
5. 抗侧信道攻击：常数时间标签比较，避免计时/功耗分析攻击；
6. 现代协议标配：TLS 1.3、IPsec、WireGuard、SSH等主流协议的默认模式。

### 关键安全注意事项
1. Nonce唯一性：**同一密钥下重复使用Nonce是致命错误**，会导致认证密钥泄露，攻击者可伪造标签；
2. 标签验证强制：解密时必须先验证标签，禁止先解密再验证（防止密文泄露）；
3. AAD一致性：加密和解密的AAD必须完全一致（包括长度和内容）；
4. 随机数质量：Nonce必须使用加密安全的随机数生成器（如os.urandom）；
5. 标签长度：生产环境禁止使用短于96位的标签，推荐128位。


> 提示：GCM的安全性依赖Nonce唯一性：即使密钥未泄露，重复Nonce也会使整个认证体系失效

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 明确GCM模式的适用场景

GCM凭借认证加密一体化、高性能、标准化的特性，成为现代密码学应用的首选模式，主要适用场景：
1. 网络协议安全：TLS/SSL 1.3、HTTPS、IPsec、WireGuard等网络传输层加密（核心场景）；
2. 存储加密：全磁盘加密（BitLocker）、数据库敏感字段加密、云存储加密；
3. API安全：REST API、微服务通信的认证加密（AAD可认证请求头，密文加密请求体）；
4. IoT/嵌入式设备：智能家居、工业物联网设备的安全通信（轻量化+并行处理）；
5. 区块链/分布式系统：交易数据加密+完整性认证，防止篡改；
6. 金融支付：移动支付、银行卡交易的端到端加密（需同时保证机密性和防篡改）；
7. 云计算：云原生应用、容器通信的加密认证（适配并行计算架构）。
### 不适用场景
- 资源极度受限的微型设备（如8位MCU）：GF(2^128)乘法计算开销较大；
- 无法保证Nonce唯一性的场景（需改用GMAC+CTR组合）；
- 需兼容老旧系统的场景（优先选择CBC+HMAC）。


> 提示：实际工程中优先使用OpenSSL/密码学库的原生GCM实现，避免自行实现伽罗华乘法（易引入安全漏洞）

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 实现GCM模式的核心加解密类

基于Python实现GCM模式的认证加密逻辑，继承分组密码模式基类，核心模块拆解：
1. 初始化方法：
   - 强制128位（16字节）分组长度，GCM仅支持该规格；
   - 设置模式名称为"GCM"；
2. 核心辅助方法：
   a. `_ghash`：Galois域哈希函数，处理AAD+密文生成认证摘要；
      - 数据填充至分组边界，初始化零块结果；
      - 逐分组异或后执行Galois乘法；
   b. `_galois_multiply`：GF(2^128)多项式乘法（简化实现，生产环境需用标准实现）；
   c. `_increment_counter`：128位计数器递增（大端序，模2^128）；
   d. `_constant_time_compare`：常数时间比较标签，防止计时攻击；
3. 加密方法`encrypt`：
   - 生成/验证96位Nonce，计算哈希密钥H=E_K(零块)；
   - 初始化计数器J0（96位Nonce+00000001）；
   - 执行CTR加密生成密文；
   - 构建认证数据（AAD长度+密文长度+AAD+密文）；
   - 计算GHASH值S，异或E_K(J0)生成128位标签；
   - 返回（密文，标签）；
4. 解密方法`decrypt`：
   - 验证Nonce/标签长度，计算哈希密钥H；
   - 重新计算认证标签并与传入标签常数时间比较（不匹配则抛异常）；
   - 标签验证通过后执行CTR解密，返回明文。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
import os
import struct
from typing import List, Tuple, Callable, Optional
from modes_base import BlockCipherMode, SimpleAES

class GCMMode(BlockCipherMode):
    """Galois/Counter Mode (GCM) - 认证加密模式"""
    
    def __init__(self, block_size: int = 16):
        super().__init__(block_size)
        self.name = "GCM"
        # GCM仅支持128位（16字节）分组大小
        if block_size != 16:
            raise ValueError("GCM模式只支持128位分组大小")
    
    def _ghash(self, data: bytes, h: bytes) -> bytes:
        """GHASH函数 - Galois域GF(2^128)上的乘法哈希"""
        # 填充到分组边界
        if len(data) % self.block_size != 0:
            padding_length = self.block_size - (len(data) % self.block_size)
            data += b'\x00' * padding_length
        
        result = b'\x00' * self.block_size
        # 逐分组处理
        for i in range(0, len(data), self.block_size):
            block = data[i:i + self.block_size]
            result = self._xor_blocks(result, block)
            result = self._galois_multiply(result, h)
        return result
    
    def _galois_multiply(self, x: bytes, y: bytes) -> bytes:
        """GF(2^128)域上的多项式乘法（简化实现）"""
        x_int = int.from_bytes(x, 'big')
        y_int = int.from_bytes(y, 'big')
        
        result = 0
        for i in range(128):
            if (y_int >> i) & 1:
                result ^= (x_int << i)
        
        # 模AES不可约多项式 x^128 + x^7 + x^2 + x + 1
        result = result & ((1 << 128) - 1)
        return result.to_bytes(16, 'big')
    
    def _generate_nonce(self) -> bytes:
        """生成推荐的96位（12字节）随机nonce"""
        return os.urandom(12)
    
    def encrypt(self, plaintext: bytes, key: bytes, iv: Optional[bytes] = None, 
                aad: bytes = b'') -> Tuple[bytes, bytes]:
        """GCM加密：返回(密文, 认证标签)"""
        cipher = SimpleAES(key)
        
        # 处理Nonce（推荐12字节）
        if iv is None:
            nonce = self._generate_nonce()
        else:
            nonce = iv
            if len(nonce) != 12:
                raise ValueError("GCM nonce长度必须为12字节")
        
        # 计算哈希密钥H = E_K(0^128)
        zero_block = b'\x00' * self.block_size
        h = cipher.encrypt_block(zero_block)
        
        # 初始化计数器J0
        j0 = nonce + b'\x00\x00\x00\x01'
        
        # CTR模式加密
        ciphertext = b''
        counter = j0
        for i in range(0, len(plaintext), self.block_size):
            keystream_block = cipher.encrypt_block(counter)
            plaintext_block = plaintext[i:i + self.block_size]
            ciphertext_block = self._xor_blocks(plaintext_block, keystream_block[:len(plaintext_block)])
            ciphertext += ciphertext_block
            counter = self._increment_counter(counter)
        
        # 计算认证标签
        aad_len = len(aad) * 8  # 转换为位
        ciphertext_len = len(ciphertext) * 8
        auth_data = struct.pack('>QQ', aad_len, ciphertext_len) + aad + ciphertext
        s = self._ghash(auth_data, h)
        tag = self._xor_blocks(s, cipher.encrypt_block(j0))
        
        return ciphertext, tag
    
    def decrypt(self, ciphertext: bytes, key: bytes, tag: bytes, 
                iv: Optional[bytes] = None, aad: bytes = b'') -> bytes:
        """GCM解密：先验证标签，再解密"""
        cipher = SimpleAES(key)
        
        # 验证输入参数
        if iv is None:
            raise ValueError("GCM解密需要提供nonce")
        if len(iv) != 12 or len(tag) != 16:
            raise ValueError("Nonce必须12字节，标签必须16字节")
        
        # 计算哈希密钥H
        zero_block = b'\x00' * self.block_size
        h = cipher.encrypt_block(zero_block)
        
        # 验证认证标签
        j0 = iv + b'\x00\x00\x00\x01'
        aad_len = len(aad) * 8
        ciphertext_len = len(ciphertext) * 8
        auth_data = struct.pack('>QQ', aad_len, ciphertext_len) + aad + ciphertext
        s = self._ghash(auth_data, h)
        expected_tag = self._xor_blocks(s, cipher.encrypt_block(j0))
        
        # 常数时间比较，防止计时攻击
        if not self._constant_time_compare(tag, expected_tag):
            raise ValueError("认证失败：标签不匹配")
        
        # CTR模式解密
        plaintext = b''
        counter = j0
        for i in range(0, len(ciphertext), self.block_size):
            keystream_block = cipher.encrypt_block(counter)
            ciphertext_block = ciphertext[i:i + self.block_size]
            plaintext_block = self._xor_blocks(ciphertext_block, keystream_block[:len(ciphertext_block)])
            plaintext += plaintext_block
            counter = self._increment_counter(counter)
        
        return plaintext
    
    def _increment_counter(self, counter: bytes) -> bytes:
        """128位计数器递增（大端序）"""
        counter_int = int.from_bytes(counter, byteorder='big')
        counter_int = (counter_int + 1) % (2 ** 128)
        return counter_int.to_bytes(16, byteorder='big')
    
    def _constant_time_compare(self, a: bytes, b: bytes) -> bool:
        """常数时间比较，避免计时攻击"""
        if len(a) != len(b):
            return False
        result = 0
        for x, y in zip(a, b):
            result |= x ^ y
        return result == 0

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 测试GCM模式的认证加密正确性

编写完整测试代码，验证GCM模式的核心功能：
1. 基础加解密验证：
   - 使用12字节固定Nonce（便于演示）、16字节AES密钥；
   - 加密测试明文，输出密文长度、标签长度、密文片段、标签；
   - 解密并验证明文一致性；
2. AAD附加认证数据测试：
   - 传入自定义AAD数据（如"Additional authenticated data"）；
   - 加密生成带AAD的密文和标签；
   - 解密时传入相同AAD，验证明文一致性；
3. 异常场景测试（可选）：
   - 篡改标签后解密，验证认证失败；
   - 更改AAD后解密，验证认证失败；
   - 使用非12字节Nonce，验证参数校验；
4. 核心验证点：
   - 标签长度固定为16字节；
   - 密文长度与明文一致（无填充）；
   - 标签验证失败时无法解密。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# GCM模式测试代码
key = b'0123456789abcdef'  # 16字节AES密钥
plaintext = b'Hello, this is a test message for block cipher modes!'  # 53字节明文

try:
    # 初始化GCM模式
    gcm_mode = GCMMode()
    
    # 1. 基础加解密测试（固定Nonce便于演示）
    nonce = b'012345678901'  # 12字节Nonce
    ciphertext, tag = gcm_mode.encrypt(plaintext, key, iv=nonce)
    print(f"密文长度: {len(ciphertext)} 字节")
    print(f"认证标签长度: {len(tag)} 字节")
    print(f"密文: {ciphertext.hex()[:64]}...")
    print(f"标签: {tag.hex()}")
    
    # 解密验证
    decrypted = gcm_mode.decrypt(ciphertext, key, tag, iv=nonce)
    print(f"解密结果: {decrypted}")
    print(f"基础测试正确性: {'✅' if plaintext == decrypted else '❌'}")
    
    # 2. AAD附加认证数据测试
    aad = b"Additional authenticated data"  # 附加认证数据（如协议头）
    ciphertext_aad, tag_aad = gcm_mode.encrypt(plaintext, key, iv=nonce, aad=aad)
    decrypted_aad = gcm_mode.decrypt(ciphertext_aad, key, tag_aad, iv=nonce, aad=aad)
    print(f"AAD认证测试: {'✅' if plaintext == decrypted_aad else '❌'}")
    
    # 3. 标签篡改测试（验证认证失败）
    fake_tag = tag[:7] + b'\xff' + tag[8:]
    try:
        gcm_mode.decrypt(ciphertext, key, fake_tag, iv=nonce)
        print(f"标签篡改测试: ❌ 认证未失败")
    except ValueError as e:
        print(f"标签篡改测试: ✅ {e}")
        
except Exception as e:
    print(f"❌ 测试错误: {e}")

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 总结GCM模式的使用规范与工程实践

### 核心使用规范
1. Nonce管理（最高优先级）：
   - 必须使用加密安全的随机数生成器生成96位Nonce；
   - 同一密钥下Nonce绝对不能重复（建议密钥轮换机制）；
   - 禁止使用可预测的Nonce（如时间戳、自增序列）；
2. 标签处理：
   - 标签长度固定为128位（16字节），禁止缩短；
   - 解密时必须先验证标签，验证失败立即终止流程；
   - 标签需与密文一起存储/传输，不可单独丢弃；
3. AAD使用：
   - AAD包含需要认证但无需加密的数据（如协议版本、请求头）；
   - 加密和解密的AAD必须完全一致（字节级匹配）；
4. 实现规范：
   - 生产环境优先使用成熟密码库（如OpenSSL、cryptography）的GCM实现；
   - 自定义实现需严格遵循GF(2^128)乘法标准，禁止简化；
   - 标签比较必须使用常数时间算法，防止计时攻击。
### 工程实践建议
- 密钥管理：使用密钥派生函数（KDF）生成GCM密钥，定期轮换；
- 性能优化：利用并行计算加速CTR加密和GHASH认证；
- 兼容性：TLS 1.3中GCM是默认模式，需兼容96位Nonce规范；
- 监控：记录Nonce使用情况，防止重复使用。


> 提示：记住GCM的核心口诀："CTR加密保机密，Galois认证保完整，Nonce唯一保安全，标签验证先于解密"

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
